# Contrasting Runs

Often when we run PassengerSim, we're not really interested in just the result of
a simulation run -- we are interested in how the results *change* across two or more
slightly different runs.  For this kind of analysis, PassengerSim offers a
[`Contrast`](../../API/contrast.html#passengersim.contrast.Contrast) tool, which 
bundles multiple 
[`SimulationTables`](../../API/summary.html#passengersim.summary.SimulationTables)
objects together, and provides comparison figures mirroring many of the single-simulation
visualizations.

In [ ]:
import passengersim as pax
from passengersim.contrast import Contrast

summary0 = pax.from_file(pax.demo_output("US25/baseline"))
summary1 = pax.from_file(pax.demo_output("US25/high-demand"))

comps = Contrast({"Baseline": summary0, "High Demand": summary1})

In [ ]:
import pathlib

from passengersim.tracers.bid_price import LegBidPriceTracer, PathBidPriceTracer

if not pathlib.Path("temp_output/3MKT-baseline.pxsim").exists():
    cfg = pax.Config.from_yaml("../../Tutorials/3MKT/network/08-untrunc-em.yaml")
    cfg.simulation_controls.num_trials = 1
    cfg.simulation_controls.num_samples = 900
    cfg.simulation_controls.show_progress_bar = False
    cfg.db = None
    cfg.outputs.reports.clear()
    cfg.carriers.AL1.rm_system = "U"
    cfg.carriers.AL2.rm_system = "P"
    cfg.carriers.AL1.rm_system_options = {}
    cfg.carriers.AL2.rm_system_options = {}
    sim = pax.Simulation(cfg)
    bp_tracer = PathBidPriceTracer(path_ids=[1, 5, 9])
    bp_tracer.attach(sim)
    leg_bp_tracer = LegBidPriceTracer(leg_ids=[101, 111])
    leg_bp_tracer.attach(sim)
    summary_3mkt = sim.run()
    summary_3mkt.to_file("temp_output/3MKT-baseline.pxsim", add_timestamp_ext=False)
else:
    summary_3mkt = pax.from_file("temp_output/3MKT-baseline.pxsim")

if not pathlib.Path("temp_output/3MKT-UU.pxsim").exists():
    cfg = pax.Config.from_yaml("../../Tutorials/3MKT/network/08-untrunc-em.yaml")
    cfg.simulation_controls.num_trials = 1
    cfg.simulation_controls.num_samples = 900
    cfg.simulation_controls.show_progress_bar = False
    cfg.db = None
    cfg.outputs.reports.clear()
    cfg.carriers.AL1.rm_system = "U"
    cfg.carriers.AL2.rm_system = "U"
    cfg.carriers.AL1.rm_system_options = {}
    cfg.carriers.AL2.rm_system_options = {}
    sim = pax.Simulation(cfg)
    bp_tracer = PathBidPriceTracer(path_ids=[1, 5, 9])
    bp_tracer.attach(sim)
    leg_bp_tracer = LegBidPriceTracer(leg_ids=[101, 111])
    leg_bp_tracer.attach(sim)
    summary_3mkt_uu = sim.run()
    summary_3mkt_uu.to_file("temp_output/3MKT-UU.pxsim", add_timestamp_ext=False)
else:
    summary_3mkt_uu = pax.from_file("temp_output/3MKT-UU.pxsim")

comps_3mkt = Contrast({"U/P": summary_3mkt, "U/U": summary_3mkt_uu})

## Carrier Revenues

[`Contrast.fig_carrier_revenues`](../../API/contrast.html#passengersim.contrast.fig_carrier_revenues).

Display the average revenues by carrier.

In [ ]:
comps.fig_carrier_revenues()

## Carrier Load Factors

[`Contrast.fig_carrier_load_factors`](../../API/contrast.html#passengersim.contrast.fig_carrier_load_factors).

Display the average system load factors by carrier.  The system load factor
is calculated based on ASM and RPM figures for the carrier, which weights 
larger capacity vehicles and longer distance legs more heavily, to reflect
their larger relative importance in evaluating carrier performance.

In [ ]:
comps.fig_carrier_load_factors()

In [ ]:
comps.fig_carrier_load_factors(load_measure="avg_leg_lf")

## Carrier Yields

[`Contrast.fig_carrier_yields`](../../API/contrast.html#passengersim.contrast.fig_carrier_yields).

Display the average yield (revenue per passenger mile) by carrier.

In [ ]:
comps.fig_carrier_yields()

## Fare Class Mix

[`Contrast.fig_fare_class_mix`](../../API/contrast.html#passengersim.contrast.fig_fare_class_mix).

Display the fare class mix by carrier.

In [ ]:
comps.fig_fare_class_mix()

## Bookings by Timeframe

[`Contrast.fig_bookings_by_timeframe`](../../API/contrast.html#passengersim.contrast.fig_bookings_by_timeframe).

Display the bookings by timeframe, optionally filtered by carrier and broken down by fare class.

In [ ]:
comps.fig_bookings_by_timeframe(by_carrier="Bison")

In [ ]:
comps.fig_bookings_by_timeframe(by_carrier="Bison", by_class=True, source_labels=True)

## Demand to Come

[`Contrast.fig_demand_to_come`](../../API/contrast.html#passengersim.contrast.fig_demand_to_come).

Display the demand to come (total demand, not just bookings) over time, showing how anticipated demand evolves as the departure date approaches.  This figure can show the mean, standard deviation, or both.

In [ ]:
comps.fig_demand_to_come(["mean", "std"])

## Bid Prices

[`Contrast.fig_bid_price_history`](../../API/contrast.html#passengersim.contrast.fig_bid_price_history).

Display the bid price history over time, showing how bid prices evolve as the departure date approaches across different simulation scenarios.

In [ ]:
comps_3mkt.fig_bid_price_history()

## Displacement Costs

[`Contrast.fig_displacement_history`](../../API/contrast.html#passengersim.contrast.fig_displacement_history).

Display the displacement cost history over time, showing how displacement costs evolve as the departure date approaches across different simulation scenarios.  Note that not all bid price based RM algorithms compute a displacement cost, in which case these values will be reported as zeros.

In [ ]:
comps_3mkt.fig_displacement_history()